In [4]:
import zipfile
from pathlib import Path

import pandas as pd
import requests

ANO = 2025
URL = f"https://www.bcb.gov.br/pda/desig/scrdata_{ANO}.zip"
CAMINHO_ZIP = Path("data/raw") / f"scrdata_{ANO}.zip"

# 1. Baixa o arquivo, se ainda não existir
if not CAMINHO_ZIP.exists():
    CAMINHO_ZIP.parent.mkdir(parents=True, exist_ok=True)
    temporario = CAMINHO_ZIP.with_suffix(".zip.part")
    print(f"Baixando {URL} (pode demorar alguns minutos)...")
    try:
        with requests.get(URL, stream=True, timeout=60, headers={"User-Agent": "Mozilla/5.0"}) as resposta:
            resposta.raise_for_status()
            with open(temporario, "wb") as arquivo:
                for bloco in resposta.iter_content(chunk_size=1024 * 1024):
                    arquivo.write(bloco)
    except requests.RequestException as erro:
        temporario.unlink(missing_ok=True)
        raise RuntimeError(f"Falha no download: {erro}") from erro
    if not zipfile.is_zipfile(temporario):
        temporario.unlink(missing_ok=True)
        raise RuntimeError("O arquivo baixado não é um ZIP válido.")
    temporario.replace(CAMINHO_ZIP)

print(f"Arquivo: {CAMINHO_ZIP} ({CAMINHO_ZIP.stat().st_size / 1e6:.0f} MB)")

# 2. Lista os CSVs e o tamanho descompactado de cada um
with zipfile.ZipFile(CAMINHO_ZIP) as zf:
    infos = sorted(
        (i for i in zf.infolist() if i.filename.lower().endswith(".csv")),
        key=lambda i: i.filename,
    )
for info in infos:
    print(f"  {info.filename}: {info.file_size / 1e6:.0f} MB descompactado")

# 3. Lê só 5 linhas do último mês, como texto, para ver o formato
mes = infos[-1].filename
for codificacao in ("utf-8", "latin-1"):
    try:
        with zipfile.ZipFile(CAMINHO_ZIP) as zf, zf.open(mes) as f:
            amostra = pd.read_csv(f, sep=";", nrows=5, dtype=str, encoding=codificacao)
        break
    except UnicodeDecodeError:
        continue

amostra.T

Baixando https://www.bcb.gov.br/pda/desig/scrdata_2025.zip (pode demorar alguns minutos)...
Arquivo: data\raw\scrdata_2025.zip (180 MB)
  scrdata_202501.csv: 101 MB descompactado
  scrdata_202502.csv: 101 MB descompactado
  scrdata_202503.csv: 102 MB descompactado
  scrdata_202504.csv: 103 MB descompactado
  scrdata_202505.csv: 104 MB descompactado
  scrdata_202506.csv: 104 MB descompactado
  scrdata_202507.csv: 100 MB descompactado
  scrdata_202508.csv: 100 MB descompactado
  scrdata_202509.csv: 100 MB descompactado
  scrdata_202510.csv: 100 MB descompactado
  scrdata_202511.csv: 100 MB descompactado
  scrdata_202512.csv: 100 MB descompactado


,0,1,2,3,4
data_base,2025-12-31,2025-12-31,2025-12-31,2025-12-31,2025-12-31
uf,AC,AC,AC,AC,AC
segmento,Arrendamento,Arrendamento,Arrendamento,Arrendamento,Banco
cliente,PJ,PJ,PJ,PJ,PF
cnae_ocupacao,Comércio; reparação de veículos automotores e ...,Construção,Indústrias de transformação,Informação e comunicação,Aposentado/pensionista
porte,Médio,Médio,Médio,Médio,Acima de 20 salários mínimos
modalidade,Operações de arrendamento,Operações de arrendamento,Operações de arrendamento,Operações de arrendamento,Empréstimos
submodalidade,Arrendamento financeiro exceto veículos automo...,Arrendamento financeiro exceto veículos automo...,Arrendamento financeiro exceto veículos automo...,Arrendamento financeiro exceto veículos automo...,"Cartão de crédito - compra, fatura parcelada o..."
origem,Sem destinação específica,Sem destinação específica,Sem destinação específica,Sem destinação específica,Sem destinação específica
indexador,Prefixado,Prefixado,Prefixado,Prefixado,Prefixado


In [5]:
prefixos_valor = ("a_vencer", "vencido", "carteira", "ativo")
colunas_valor = [c for c in amostra.columns if c.startswith(prefixos_valor)]
colunas_texto = [c for c in amostra.columns if c not in colunas_valor and c != "numero_de_operacoes"]

# category economiza memória em colunas de texto com poucos valores distintos
tipos = {c: "category" for c in colunas_texto}
tipos.update({c: "float64" for c in colunas_valor})
tipos["numero_de_operacoes"] = "Int64"

with zipfile.ZipFile(CAMINHO_ZIP) as zf, zf.open(mes) as f:
    df = pd.read_csv(f, sep=";", decimal=",", dtype=tipos, encoding=codificacao)

print(f"Mês: {mes}")
print(f"Linhas x colunas: {df.shape}")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1e6:.0f} MB\n")

for c in colunas_texto:
    print(f"{c}: {df[c].nunique()} valores -> {list(df[c].cat.categories[:8])}")

sigilo = df["numero_de_operacoes"] == -1
print(f"\nLinhas com numero_de_operacoes = -1: {sigilo.sum()} de {len(df)} ({sigilo.mean():.1%})")
print(f"Parcela da carteira nessas linhas: {df.loc[sigilo, 'carteira_ativa'].sum() / df['carteira_ativa'].sum():.1%}")
print(f"Carteira ativa total: {df['carteira_ativa'].sum():,.0f}\n")

taxa = df.groupby("cliente", observed=True)[["carteira_inadimplencia", "carteira_ativa"]].sum()
taxa["inadimplencia_%"] = 100 * taxa["carteira_inadimplencia"] / taxa["carteira_ativa"]
taxa

Mês: scrdata_202512.csv
Linhas x colunas: (310452, 24)
Memória: 38 MB

data_base: 1 valores -> ['2025-12-31']
uf: 27 valores -> ['AC', 'AL', 'AM', 'AP', 'BA', 'CE', 'DF', 'ES']
segmento: 8 valores -> ['Arrendamento', 'Banco', 'Cooperativa', 'Desenvolvimento/Fomento', 'Financeira', 'Fintech', 'Instituição de pagamento', 'Outros']
cliente: 2 valores -> ['PF', 'PJ']
cnae_ocupacao: 30 valores -> ['Administração pública, defesa e seguridade social', 'Agricultura, pecuária, produção florestal, pesca e aquicultura', 'Alojamento e alimentação', 'Aposentado/pensionista', 'Artes, cultura, esporte e recreação', 'Atividades administrativas e serviços complementares', 'Atividades financeiras, de seguros e serviços relacionados', 'Atividades imobiliárias']
porte: 13 valores -> ['Acima de 20 salários mínimos', 'Até 1 salário mínimo', 'Grande', 'Indisponível', 'Mais de 1 a 2 salários mínimos', 'Mais de 10 a 20 salários mínimos', 'Mais de 2 a 3 salários mínimos', 'Mais de 3 a 5 salários mínimos']
modal

,carteira_inadimplencia,carteira_ativa,inadimplencia_%
cliente,,,
PF,2.328482e+11,4.525137e+12,5.14566
PJ,7.270723e+10,2.919184e+12,2.49067
